<a href="https://colab.research.google.com/github/msrehman786/IBM-AI-Certification/blob/main/Translator_Babel_Fish_(Language_Translator)_with_LLM%2C_STT%2C_%26_TTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!python -m venv my_env
!source my_env/bin/activate # activate my_env

Error: Command '['/content/my_env/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.
/bin/bash: line 1: my_env/bin/activate: No such file or directory


In [2]:
!git clone https://github.com/ibm-developer-skills-network/translator-with-voice-and-watsonx
!cd translator-with-voice-and-watsonx
!cp -r * ../.

Cloning into 'translator-with-voice-and-watsonx'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 74 (delta 8), reused 4 (delta 4), pack-reused 61 (from 1)
Receiving objects: 100% (74/74), 29.73 KiB | 9.91 MiB/s, done.
Resolving deltas: 100% (23/23), done.


In [10]:
!pip install flask flask_cors requests ibm_watson_machine_learning


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: --reinstall


In [4]:
!pip install ibm_watsonx_ai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 13.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 52.1 MB/s eta 0:00:00
  Created wheel for ibm-cos-sdk: filename=ibm_cos_sdk-2.14.3-py3-none-any.whl size=77232 sha256=30b4e1c7fdc081f1d0ccc12b740c11165389ec450ef5b23b61fe16c9a53d54f6
  Stored in directory: /root/.cache/pip/wheels/a2/99/26/abae30f0704d74b848aac9aac73e9b47977b808a75a5a984fe
  Created wheel for ibm-cos-sdk-core: filename=ibm_cos_sdk_core-2.14.3-py3-none-any.whl size=662101 sha256=c076fedfbb0b4af3ca49e5185cac050aa1a5258d55bc6e16b3204412f5322562
  Stored in directory: /root/.cache/pip/wheels/47/62/69/1e99fea95b82e8dbd4ea97e78160224269774a43883f908551
  Created whe

In [5]:
!curl "https://github.com/watson-developer-cloud/doc-tutorial-downloads/raw/master/speech-to-text/0001.flac" -sLo example.flac

worker.py

In [6]:
# To call watsonx's LLM, we need to import the library of IBM watsonx.ai
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
from ibm_watsonx_ai.foundation_models import ModelInference
# Define the model parameters
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import DecodingMethods
from ibm_watsonx_ai import Credentials
from ibm_watsonx_ai.foundation_models.schema import TextChatParameters
from google.colab import userdata


# placeholder for Watsonx_API and Project_id incase you need to use the code outside this environment
# API_KEY = "Your WatsonX API"
PROJECT_ID= "5f141825-b9a2-4a7a-a3af-3ddd1b7006d9"

# Define the credentials
credentials = Credentials(
    url="https://eu-gb.ml.cloud.ibm.com",
    api_key=userdata.get('WATSONX_API_KEY')
)

# Specify model_id that will be used for inferencing
model_id = "mistralai/mistral-small-3-1-24b-instruct-2503"

parameters = TextChatParameters(
    temperature=0,
    max_tokens=1024
)

# Define the LLM

model = ModelInference(
    model_id=model_id,
    params=parameters,
    credentials=credentials,
    project_id=PROJECT_ID
)

import requests

#The speech_to_text function will take in audio data as a parameter, make an API call to the Watson Speech-to-Text API using the requests library, and return the transcription of the audio data.
def speech_to_text(audio_binary):

	# Set up Watson Speech-to-Text HTTP Api url
	base_url = 'https://sn-watson-stt.labs.skills.network'
	api_url = base_url+'/speech-to-text/api/v1/recognize'

	# Set up parameters for our HTTP reqeust
	params = {
		'model': 'en-US_Multimedia',
	}

	# Set up the body of our HTTP request
	body = audio_binary

	# Send a HTTP Post request
	response = requests.post(api_url, params=params, data=audio_binary).json()

	# Parse the response to get our transcribed text
	text = 'null'
	while bool(response.get('results')):
		print('Speech-to-Text response:', response)
		text = response.get('results').pop().get('alternatives').pop().get('transcript')
		print('recognised text: ', text)
		return text

def text_to_speech(text, voice=""):
  # Set up Watson Text-to-Speech HTTP Api url
  base_url = 'https://sn-watson-tts.labs.skills.network'
  api_url = base_url + '/text-to-speech/api/v1/synthesize?output=output_text.wav'

	# Adding voice parameter in api_url if the user has selected a preferred voice
  if voice != "" and voice != "default":
    api_url += "&voice=" + voice

	# Set the headers for our HTTP request
  headers = {
    'Accept': 'audio/wav',
    'Content-Type': 'application/json',
  }

	# Set the body of our HTTP request
  json_data = {
    'text': text,
  }

	# Send a HTTP Post reqeust to Watson Text-to-Speech Service
  response = requests.post(api_url, headers=headers, json=json_data)
  print('Text-to-Speech response:', response)
  return response.content


#Watsonx process message function
##We will be updating the function called watsonx_process_message, which will take in a prompt and pass it to Watsonx's mistralai/mistral-medium-2505 API to receive a response. Essentially, it's the equivalent of pressing the send button to get a response from ChatGPT.f
##Go ahead and update the watsonx_process_message function in the worker.py file with the following.
def watsonx_process_message(user_message):
    prompt = f"""Respond to the query: ```{user_message}```"""
    messages = [{"role": "user", "content": prompt}]
    response = model.chat(messages=messages)
    response_text = response["choices"][0]["message"]["content"]
    print("watsonx response:", response_text)
    return response_text.strip()

In [ ]:
!curl https://sn-watson-stt.labs.skills.network/speech-to-text/api/v1/models

^C


In [ ]:
!curl "https://sn-watson-stt.labs.skills.network/speech-to-text/api/v1/recognize" --header "Content-Type: audio/flac" --data-binary @example.flac

In [ ]:
!curl https://sn-watson-tts.labs.skills.network/text-to-speech/api/v1/voices

^C


In [ ]:
!curl "https://sn-watson-tts.labs.skills.network/text-to-speech/api/v1/synthesize" --header "Content-Type: application/json" --data '{"text":"Hello world"}' --header "Accept: audio/wav" --output output.wav

In [ ]:
!curl "https://sn-watson-tts.labs.skills.network/text-to-speech/api/v1/synthesize?voice=es-LA_SofiaV3Voice" --header "Content-Type: application/json" --data '{"text":"Hola! Hoy es un dia muy bonito."}' --header "Accept: audio/mp3" --output hola.mp3

server.py

In [13]:
!pip install flask pyngrok flask_cors

In [14]:
import base64
import json
from flask import Flask, render_template, request
from flask_cors import CORS
import os
from worker import speech_to_text, text_to_speech, watsonx_process_message
from pyngrok import ngrok

app = Flask(__name__)
cors = CORS(app, resources={r"/*": {"origins": "*"}})


# Optional: Set ngrok auth token (skip if not using one)
ngrok.set_auth_token("3IBE5FhsNRF52VxuQwh0JnlsAqu_6RoqvbJKa7EwEAespyiP")  # Replace with your token

# Open a ngrok tunnel to port 5000 (where Flask will run)
public_url = ngrok.connect(5001).public_url
print(f"✅ Flask app is live at: {public_url}")


# Define the route for the index page
@app.route('/', methods=['GET'])
def index():
    return render_template('index.html')  # Render the index.html template


@app.route('/speech-to-text', methods=['POST'])
def speech_to_text_route():
    print("processing Speech-to-Text")
    audio_binary = request.data # Get the user's speech from their request
    text = speech_to_text(audio_binary) # Call speech_to_text function to transcribe the speech

	# Return the response to user in JSON format
    response = app.response_class(
        response=json.dumps({'text': text}),
        status=200,
        mimetype='application/json'
    )
    print(response)
    print(response.data)
    return response


@app.route('/process-message', methods=['POST'])
def process_message_route():
    user_message = request.json['userMessage'] # Get user's message from their request
    print('user_message', user_message)

    voice = request.json['voice'] # Get user\'s preferred voice from their request
    print('voice', voice)

	# Call watsonx_process_message function to process the user's message and get a response back
    watsonx_response_text = watsonx_process_message(user_message)

	# Clean the response to remove any emptylines
    watsonx_response_text = os.linesep.join([s for s in watsonx_response_text.splitlines() if s])

	# Call our text_to_speech function to convert Watsonx Api's reponse to speech
    watsonx_response_speech = text_to_speech(watsonx_response_text, voice)

    # convert watsonx_response_speech to base64 string so it can be sent back in the JSON response
    watsonx_response_speech = base64.b64encode(watsonx_response_speech).decode('utf-8')

	# Send a JSON response back to the user containing their message\'s response both in text and speech formats
    response = app.response_class(
        response=json.dumps({"watsonxResponseText": watsonx_response_text, "watsonxResponseSpeech": watsonx_response_speech}),
        status=200,
        mimetype='application/json'
    )

    print(response)
    return response


if __name__ == "__main__":
    app.run(host='0.0.0.0', port=5001)
    #app.run(port=8000, host='0.0.0.0')

✅ Flask app is live at: https://snowless-persuader-nest.ngrok-free.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [09/Sep/2026 12:50:33] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Sep/2026 12:50:34] "GET /static/style.css HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Sep/2026 12:50:34] "GET /static/script.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [09/Sep/2026 12:50:35] "GET /favicon.ico HTTP/1.1" 404 -


user_message Hi
voice en-US_OliviaV3Voice


/usr/local/lib/python3.13/dist-packages/ibm_watsonx_ai/wml_resource.py:100: WatsonxAPIWarning: This model is a Non-IBM Product governed by a third-party license that may impose use restrictions and other obligations. By using this model you agree to its terms as identified in the following URL.
ID: disclaimer_warning
More info: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-models.html?context=wx
  warn(cls._build_warning_message(warning), WatsonxAPIWarning)


watsonx response: Hello! How can I assist you today? If you're up for it, I can tell a joke to start. Here it is:

What do you call fake spaghetti?

An impasta!
user_message Hello
voice fr-CA_LouiseV3Voice
watsonx response: Hello! How can I assist you today? If you're up for it, I can tell a joke to start. Here it is:

What do you call fake spaghetti?

An impasta!
